# W4-W6: GraphSAGE + Transformer + 三路融合训练

**运行环境**: Kaggle Notebook (GPU T4/P100)  
**目标**: 依次完成 GraphSAGE 节点嵌入、Transformer 序列编码、三路 Gated Fusion 训练  

上传以下文件到 Kaggle Dataset:  
- `fraud_graph_edges.csv`
- `node_id_map.csv`
- `node_labels.npy`
- `lgb_baseline.pkl`
- `oof_preds_baseline.npy`

## 安装依赖库

- 在 Kaggle **Settings** 中打开 **Internet**。
- **`DGL backend not selected... Assuming PyTorch`** 只是提示默认后端为 PyTorch，**可忽略**。
- **`ModuleNotFoundError: torchdata.datapipes`**：来自 PyPI 的 **`dgl==2.1.0`** 仍会引用该模块；在 **PyTorch 2.10 + 新版 torchdata** 上经常不存在。请用下面代码格安装 **官方 wheel 的 DGL ≥2.4**（不再依赖 `torchdata.datapipes`）；**cu124** 的 wheel 在 **CUDA 12.8** 上通常仍可用。
- 若 wheel 与 **PyTorch 2.10** 二进制不兼容，再尝试代码格里的**备选 pip**（降级 `torchdata`；本笔记不用 torchtune 时一般可接受 pip 的冲突提示）。
- 安装或强制重装 DGL 后请 **Restart session**，再从上到下运行。

In [ ]:
# 打印版本：https://www.dgl.ai/pages/start.html
import torch
print(f"PyTorch {torch.__version__}, CUDA {torch.version.cuda}")

# 重要：Kaggle 预装包很多，直接 pip install 可能升级 numpy/pandas/scipy 引发大量“dependency conflicts”警告。
# 这里用 --no-deps：只装 DGL 本体，不改动其它依赖（更稳）。
# cu124 的 wheel 在 CUDA 12.8 上通常可用；若不兼容再换下面备选索引。
!pip install --force-reinstall --no-deps -q dgl -f https://data.dgl.ai/wheels/torch-2.6/cu124/repo.html

# 备选（二选一取消注释）：
# !pip install --force-reinstall --no-deps -q dgl -f https://data.dgl.ai/wheels/torch-2.4/cu124/repo.html
# !pip install --force-reinstall --no-deps -q dgl -f https://data.dgl.ai/wheels/torch-2.5/cu124/repo.html

# 最后手段（不推荐）：继续用 PyPI 的 dgl==2.1.0 + 旧 torchdata（可能与 Kaggle 预装 torchtune 冲突）
# !pip install -q dgl==2.1.0 "torchdata>=0.6,<0.9"

print("安装完成后请 Restart session，再从下一格开始运行。")

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import dgl
from dgl.nn import SAGEConv
from sklearn.metrics import roc_auc_score, average_precision_score
from scipy.stats import ks_2samp
import warnings, gc, pickle, os
warnings.filterwarnings('ignore')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

# Kaggle 路径：竞赛与数据集会挂载在更深层目录（/kaggle/input/competitions/... 与 /kaggle/input/datasets/...）
INPUT_ROOT = '/kaggle/input'
print('Top-level under /kaggle/input:', os.listdir(INPUT_ROOT))

def _find_dir_with_file(filename: str, max_depth: int = 5) -> str:
    # 只在 competitions 与 datasets 两棵目录下查找，避免扫太多
    roots = [
        os.path.join(INPUT_ROOT, 'competitions'),
        os.path.join(INPUT_ROOT, 'datasets'),
    ]
    roots = [r for r in roots if os.path.isdir(r)]

    for root in roots:
        for dirpath, dirnames, filenames in os.walk(root):
            rel = os.path.relpath(dirpath, root)
            depth = 0 if rel == '.' else rel.count(os.sep) + 1
            if depth > max_depth:
                dirnames[:] = []
                continue

            if filename in filenames:
                return dirpath

    raise FileNotFoundError(
        f"在 {INPUT_ROOT}/(competitions|datasets) 下找不到 {filename}。\n"
        f"请确认右侧 Input 已添加对应 Competition/Dataset，且文件名无误。"
    )

# 官方 IEEE-CIS Fraud Detection（需要包含 train_transaction.csv）
DATA_DIR = _find_dir_with_file('train_transaction.csv')
print('Using DATA_DIR:', DATA_DIR)

# 你上传的自定义 Dataset（需要包含 fraud_graph_edges.csv）
CUSTOM_DIR = _find_dir_with_file('fraud_graph_edges.csv')
print('Using CUSTOM_DIR:', CUSTOM_DIR)

## Part 1: 数据准备

In [ ]:
# 加载原始数据
def reduce_mem(df):
    for col in df.columns:
        t = df[col].dtype
        if t == np.float64: df[col] = df[col].astype(np.float32)
        elif t == np.int64:
            if df[col].min() >= np.iinfo(np.int32).min and df[col].max() <= np.iinfo(np.int32).max:
                df[col] = df[col].astype(np.int32)
    return df

train_trans = reduce_mem(pd.read_csv(f'{DATA_DIR}/train_transaction.csv'))
train_id = reduce_mem(pd.read_csv(f'{DATA_DIR}/train_identity.csv'))
train = train_trans.merge(train_id, on='TransactionID', how='left')
del train_trans, train_id; gc.collect()

# 时间 + 金额特征
train['hour'] = (train['TransactionDT'] // 3600) % 24
train['weekday'] = (train['TransactionDT'] // 86400) % 7
train['amt_log'] = np.log1p(train['TransactionAmt'])

print(f'训练数据: {train.shape}')

## Part 2: W4 — GraphSAGE 训练

In [ ]:
# 用训练集字段“重建”交易-交易图边（冲分版）
# 思路：用实体共享关系构边（card/addr/email/device 等），并对每个实体组做度数/组大小限制，避免边数爆炸

node_map_path = f'{CUSTOM_DIR}/node_id_map.csv'
node_map = pd.read_csv(node_map_path)
cols = {c.lower(): c for c in node_map.columns}

tid_col = cols.get('transactionid') or cols.get('transaction_id') or node_map.columns[0]
nid_col = cols.get('node_idx') or cols.get('node_id') or cols.get('nodeid') or cols.get('nid') or cols.get('id')
if nid_col is None:
    nid_col = node_map.columns[1] if node_map.shape[1] > 1 else node_map.columns[0]

num_nodes = int(node_map[nid_col].max()) + 1

# 先给 train 每行补上 node_idx（用于构边与后续对齐）
_tid_to_nid = dict(zip(node_map[tid_col].astype(np.int64).values, node_map[nid_col].astype(np.int64).values))
train['node_idx'] = train['TransactionID'].astype(np.int64).map(_tid_to_nid).astype('int64')
assert train['node_idx'].notna().all(), '存在 TransactionID 未映射到 node_idx'

# 选择用于构边的实体键（存在则使用）
entity_cols = []
for c in ['card1', 'addr1', 'P_emaildomain', 'R_emaildomain', 'DeviceType']:
    if c in train.columns:
        entity_cols.append(c)

print('Edge build entity cols:', entity_cols)

# 控制规模
max_group_size = 2000   # 单个实体组最多参与构边的节点数（过大组会爆边）
max_deg_per_key = 32    # 每个实体键下，每个节点最多连几条边
seed = 42
rng = np.random.RandomState(seed)

src_list = []
dst_list = []

# 将每个 entity col 转成可 group 的 code（object 列 factorize）
for col in entity_cols:
    s = train[col]
    if s.dtype == 'O' or str(s.dtype).startswith('string'):
        code, _ = pd.factorize(s.fillna('NA'), sort=False)
        code = code.astype(np.int64)
    else:
        code = s.fillna(-1).astype(np.int64).values

    nodes = train['node_idx'].values.astype(np.int64)

    # 按 code 排序，便于分组扫描
    order = np.argsort(code, kind='mergesort')
    code_s = code[order]
    nodes_s = nodes[order]

    # 逐组构边：对组内节点随机打散，做“环形+跳连”以控制每点度数
    i = 0
    n = len(nodes_s)
    kept_groups = 0
    total_edges = 0
    while i < n:
        j = i + 1
        while j < n and code_s[j] == code_s[i]:
            j += 1

        k = code_s[i]
        if k != -1:
            group = nodes_s[i:j]
            if len(group) >= 2:
                if len(group) > max_group_size:
                    group = rng.choice(group, size=max_group_size, replace=False)

                g = group.copy()
                rng.shuffle(g)
                m = len(g)

                # 为了控制度数：对每个点连向接下来 t 个点（环上）
                t = min(max_deg_per_key, m - 1)
                for step in range(1, t + 1):
                    src_list.append(g)
                    dst_list.append(np.roll(g, -step))

                kept_groups += 1
                total_edges += m * t

        i = j

    print(f'[{col}] groups_kept={kept_groups:,}, edges_added~={total_edges:,}')

src = np.concatenate(src_list).astype(np.int64)
dst = np.concatenate(dst_list).astype(np.int64)
print(f'Edges built: {len(src):,} (directed, before making undirected)')

# 构建无向图（双向边）+ self-loop
src_ud = np.concatenate([src, dst])
dst_ud = np.concatenate([dst, src])
g = dgl.graph((src_ud, dst_ud), num_nodes=num_nodes)
g = dgl.add_self_loop(g)
print(f'图: {g.num_nodes()} 节点, {g.num_edges()} 边')

# 节点特征：[in_degree, amt_log, hour, weekday]
# 用 node_id_map.csv 将 TransactionID 映射到图节点
cols = {c.lower(): c for c in node_map.columns}

tid_col = cols.get('transactionid') or cols.get('transaction_id') or node_map.columns[0]
nid_col = (
    cols.get('node_id')
    or cols.get('nodeid')
    or cols.get('nid')
    or cols.get('node')
    or cols.get('id')
    or (node_map.columns[1] if node_map.shape[1] > 1 else node_map.columns[0])
)

_tmp = train[['TransactionID', 'amt_log', 'hour', 'weekday', 'isFraud']].copy()

# z-score 标准化（缺失填 0）
for c in ['amt_log', 'hour', 'weekday']:
    _tmp[c] = _tmp[c].astype(np.float32)
    mu = float(_tmp[c].mean())
    sd = float(_tmp[c].std())
    if sd > 0:
        _tmp[c] = (_tmp[c] - mu) / sd
    else:
        _tmp[c] = 0.0
    _tmp[c] = _tmp[c].fillna(0.0)

_tid_to_row = dict(zip(_tmp['TransactionID'].astype(np.int64).values, np.arange(len(_tmp), dtype=np.int64)))

# 初始化 node_feat + node_label
in_deg = g.in_degrees().float().cpu().numpy().reshape(-1, 1)
node_feat_np = np.zeros((g.num_nodes(), 4), dtype=np.float32)
node_feat_np[:, 0:1] = in_deg

node_label_np = np.full((g.num_nodes(),), -1.0, dtype=np.float32)  # -1 表示无标签

n_filled = 0
for tid, nid in zip(node_map[tid_col].astype(np.int64).values, node_map[nid_col].astype(np.int64).values):
    row = _tid_to_row.get(int(tid))
    if row is None:
        continue
    if nid < 0 or nid >= g.num_nodes():
        continue

    node_feat_np[nid, 1] = float(_tmp.at[row, 'amt_log'])
    node_feat_np[nid, 2] = float(_tmp.at[row, 'hour'])
    node_feat_np[nid, 3] = float(_tmp.at[row, 'weekday'])
    node_label_np[nid] = float(_tmp.at[row, 'isFraud'])
    n_filled += 1

print(f'Node features shape: {node_feat_np.shape}, filled from map: {n_filled:,}/{len(node_map):,}')

node_feat = torch.tensor(node_feat_np, dtype=torch.float32)
labels = torch.tensor(node_label_np, dtype=torch.float32)
label_mask = labels >= 0
print(f'Labeled nodes: {int(label_mask.sum().item()):,}/{g.num_nodes():,}')

In [ ]:
class FraudGraphSAGE(nn.Module):
    def __init__(self, in_feats, hidden=128, out_feats=64):
        super().__init__()
        self.conv1 = SAGEConv(in_feats, hidden, 'mean')
        self.conv2 = SAGEConv(hidden, out_feats, 'mean')
        self.bn1 = nn.BatchNorm1d(hidden)
        self.drop = nn.Dropout(0.3)
        self.classifier = nn.Linear(out_feats, 1)

    def forward(self, g, x):
        h = self.drop(torch.relu(self.bn1(self.conv1(g, x))))
        h = self.conv2(g, h)
        return h

    def predict(self, g, x):
        h = self.forward(g, x)
        return self.classifier(h).squeeze(1)

model_gnn = FraudGraphSAGE(in_feats=node_feat.shape[1]).to(device)
optimizer = torch.optim.Adam(model_gnn.parameters(), lr=1e-3, weight_decay=5e-4)

# Focal Loss
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha, self.gamma = alpha, gamma
    def forward(self, pred, target):
        bce = nn.functional.binary_cross_entropy_with_logits(pred, target, reduction='none')
        pt = torch.exp(-bce)
        return (self.alpha * (1 - pt) ** self.gamma * bce).mean()

criterion = FocalLoss()

# 训练（仅在有标签的节点上）
g_gpu = g.to(device)
feat_gpu = node_feat.to(device)
labels_gpu = labels.to(device)
mask_gpu = label_mask.to(device)

# 只对有标签节点做切分
idx = torch.where(mask_gpu)[0]
perm = idx[torch.randperm(len(idx))]
cut = int(len(perm) * 0.8)
train_idx = perm[:cut]
val_idx = perm[cut:]

best_auc = 0
for epoch in range(200):
    model_gnn.train()
    logits = model_gnn.predict(g_gpu, feat_gpu)
    loss = criterion(logits[train_idx], labels_gpu[train_idx])
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 20 == 0:
        model_gnn.eval()
        with torch.no_grad():
            preds = torch.sigmoid(logits[val_idx]).cpu().numpy()
            auc = roc_auc_score(labels_gpu[val_idx].cpu().numpy(), preds)
        print(f'Epoch {epoch+1}: loss={loss.item():.4f}, val_auc={auc:.4f}')
        if auc > best_auc:
            best_auc = auc
            torch.save(model_gnn.state_dict(), 'graphsage_best.pt')

print(f'Best GNN AUC (labeled nodes): {best_auc:.4f}')

In [ ]:
# 提取 GraphSAGE Embedding
model_gnn.eval()
with torch.no_grad():
    gnn_embeddings = model_gnn(g_gpu, feat_gpu).cpu().numpy()
print(f'GNN Embeddings: {gnn_embeddings.shape}')
np.save('gnn_embeddings.npy', gnn_embeddings)

## Part 3: W5 — Behavior Transformer 训练

In [ ]:
class BehaviorTransformer(nn.Module):
    def __init__(self, in_feats: int, feat_dim=32, n_heads=4, n_layers=2, max_len=50, dropout=0.1):
        super().__init__()
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=feat_dim, nhead=n_heads,
            dim_feedforward=feat_dim*4, dropout=dropout, batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.pos_emb = nn.Embedding(max_len + 1, feat_dim)
        self.cls_token = nn.Parameter(torch.randn(1, 1, feat_dim))
        self.input_proj = nn.Linear(in_feats, feat_dim)
        self.classifier = nn.Linear(feat_dim, 1)

    def forward(self, x):
        B, S, _ = x.shape
        x = self.input_proj(x)
        pos = torch.arange(S + 1, device=x.device).unsqueeze(0)
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)
        x = x + self.pos_emb(pos)
        out = self.encoder(x)
        return out[:, 0, :]  # CLS embedding

    def predict(self, x):
        cls_emb = self.forward(x)
        return self.classifier(cls_emb).squeeze(1)

In [ ]:
# 构建交易序列（更强特征版）
SEQ_LEN = 50

# 基础特征（保证存在）
base_cols = ['amt_log', 'hour', 'weekday', 'TransactionAmt']

# 从 V/C/D 等数值列里自动挑选一些信息量更大的特征（按缺失率+方差筛选）
max_extra = 24  # 额外特征数（可调大，但越大越慢/越吃内存）

candidate_cols = []
for c in train.columns:
    if c in {'TransactionID', 'TransactionDT', 'isFraud'}:
        continue
    if c.startswith(('V', 'C', 'D')):
        candidate_cols.append(c)

# 只保留数值列
candidate_cols = [c for c in candidate_cols if pd.api.types.is_numeric_dtype(train[c])]

stats = []
for c in candidate_cols:
    s = train[c]
    miss = float(s.isna().mean())
    if miss >= 0.95:
        continue
    v = float(s.astype('float32').fillna(0).var())
    stats.append((v, c))

stats.sort(reverse=True)
extra_cols = [c for _, c in stats[:max_extra]]

FEAT_COLS = [c for c in base_cols if c in train.columns] + extra_cols
print(f'Seq features: {len(FEAT_COLS)} dims')

# 标准化到 *_norm（缺失填 0）
for col in FEAT_COLS:
    mu, sigma = train[col].mean(), train[col].std()
    if sigma and sigma > 0:
        train[col + '_norm'] = ((train[col] - mu) / sigma).astype(np.float32)
    else:
        train[col + '_norm'] = 0.0
    train[col + '_norm'] = train[col + '_norm'].fillna(0.0)

norm_cols = [c + '_norm' for c in FEAT_COLS]

# 分组实体键：优先 card1+addr1（更细粒度），否则退回 card1
if 'addr1' in train.columns:
    seq_group_cols = ['card1', 'addr1']
else:
    seq_group_cols = ['card1']

print('Seq group key:', '+'.join(seq_group_cols))
print('构建序列...')
sequences = []
seq_labels = []
seq_keys = []  # 与 seq_embeddings 行一一对应

# 为了 key 可哈希，用字符串拼接（避免 tuple 中 NaN 的坑）
if seq_group_cols == ['card1', 'addr1']:
    _key_series = train['card1'].astype('int64').astype('string') + '|' + train['addr1'].fillna(-1).astype('int64').astype('string')
    train['_seq_key'] = _key_series
    group_iter = train.groupby('_seq_key')
else:
    train['_seq_key'] = train['card1']
    group_iter = train.groupby('_seq_key')

for k, group in group_iter:
    group = group.sort_values('TransactionDT')
    feats = group[norm_cols].values
    label = group['isFraud'].values[-1]

    if len(feats) >= 3:
        feats = feats[-SEQ_LEN:]
        if len(feats) < SEQ_LEN:
            pad = np.zeros((SEQ_LEN - len(feats), len(norm_cols)), dtype=np.float32)
            feats = np.vstack([pad, feats])
        sequences.append(feats)
        seq_labels.append(label)
        seq_keys.append(k)

X_seq = np.array(sequences, dtype=np.float32)
y_seq = np.array(seq_labels, dtype=np.float32)
print(f'序列数据: {X_seq.shape}, 欺诈率: {y_seq.mean():.4f}')

In [ ]:
# 训练 Transformer
from torch.utils.data import TensorDataset, DataLoader

split = int(len(X_seq) * 0.8)
train_ds = TensorDataset(torch.tensor(X_seq[:split]), torch.tensor(y_seq[:split]))
val_ds = TensorDataset(torch.tensor(X_seq[split:]), torch.tensor(y_seq[split:]))
train_dl = DataLoader(train_ds, batch_size=256, shuffle=True)
val_dl = DataLoader(val_ds, batch_size=512)

model_seq = BehaviorTransformer(in_feats=X_seq.shape[2], feat_dim=32, n_heads=4, n_layers=2).to(device)
opt_seq = torch.optim.Adam(model_seq.parameters(), lr=1e-3)
focal = FocalLoss()

best_seq_auc = 0
for epoch in range(30):
    model_seq.train()
    total_loss = 0
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        pred = model_seq.predict(xb)
        loss = focal(pred, yb)
        opt_seq.zero_grad()
        loss.backward()
        opt_seq.step()
        total_loss += loss.item()

    model_seq.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for xb, yb in val_dl:
            preds = torch.sigmoid(model_seq.predict(xb.to(device))).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(yb.numpy())
    val_auc = roc_auc_score(all_labels, all_preds)
    print(f'Epoch {epoch+1}: loss={total_loss/len(train_dl):.4f}, val_auc={val_auc:.4f}')
    if val_auc > best_seq_auc:
        best_seq_auc = val_auc
        torch.save(model_seq.state_dict(), 'transformer_best.pt')

print(f'Best Transformer AUC: {best_seq_auc:.4f}')

In [ ]:
# 提取 Transformer CLS Embeddings
model_seq.eval()
all_embs = []
all_dl = DataLoader(TensorDataset(torch.tensor(X_seq)), batch_size=512)
with torch.no_grad():
    for (xb,) in all_dl:
        emb = model_seq(xb.to(device)).cpu().numpy()
        all_embs.append(emb)
seq_embeddings = np.vstack(all_embs)
print(f'Transformer Embeddings: {seq_embeddings.shape}')
np.save('seq_embeddings.npy', seq_embeddings)

## Part 4: W6 — 三路 Gated Fusion

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

class ResidualFusion(nn.Module):
    """Predict a delta-logit to add on top of baseline logit."""
    def __init__(self, feat_dim: int, hidden_dim: int = 128):
        super().__init__()
        self.gate = nn.Sequential(nn.Linear(feat_dim, feat_dim), nn.Sigmoid())
        self.mlp = nn.Sequential(
            nn.Linear(feat_dim, hidden_dim), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(hidden_dim, hidden_dim // 2), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(hidden_dim // 2, 1)
        )

    def forward(self, feats):
        feats = feats * self.gate(feats)
        return self.mlp(feats).squeeze(1)  # delta logit


def _align_gnn_to_train(train_df: pd.DataFrame, gnn_emb: np.ndarray, custom_dir: str):
    """返回与 train_df 行数一致的 gnn embedding + 缺失行数。"""
    if gnn_emb.shape[0] == len(train_df):
        return gnn_emb, 0

    map_path = os.path.join(custom_dir, 'node_id_map.csv')
    if not os.path.exists(map_path):
        raise FileNotFoundError(
            f"gnn_embeddings 行数({gnn_emb.shape[0]})与 train 行数({len(train_df)})不一致，且找不到 node_id_map.csv：{map_path}"
        )

    m = pd.read_csv(map_path)
    cols = {c.lower(): c for c in m.columns}

    tid_col = cols.get('transactionid') or cols.get('transaction_id')
    nid_col = (
        cols.get('node_id')
        or cols.get('nodeid')
        or cols.get('nid')
        or cols.get('node')
        or cols.get('id')
    )

    if tid_col is None:
        if m.shape[1] >= 2:
            tid_col = m.columns[0]
        else:
            raise ValueError(f'node_id_map.csv 列太少，无法推断 TransactionID 列：{list(m.columns)}')

    if nid_col is None:
        if m.shape[1] >= 2:
            nid_col = m.columns[1]
        else:
            raise ValueError(f'node_id_map.csv 列太少，无法推断 node_id 列：{list(m.columns)}')

    tid_to_nid = dict(zip(m[tid_col].astype(np.int64).values, m[nid_col].astype(np.int64).values))
    n_missing = 0

    out = np.zeros((len(train_df), gnn_emb.shape[1]), dtype=np.float32)
    for i, tid in enumerate(train_df['TransactionID'].astype(np.int64).values):
        nid = tid_to_nid.get(int(tid))
        if nid is None or nid < 0 or nid >= gnn_emb.shape[0]:
            n_missing += 1
            continue
        out[i] = gnn_emb[nid]

    return out, n_missing


def _align_seq_to_train(train_df: pd.DataFrame, seq_emb: np.ndarray, seq_keys):
    """返回与 train_df 行数一致的 seq embedding + 缺失行数。

    seq_keys: 来自 W5 的 seq_keys（字符串 key）。
    """
    if len(seq_keys) != seq_emb.shape[0]:
        raise ValueError(f'seq_keys 长度({len(seq_keys)})与 seq_embeddings 行数({seq_emb.shape[0]})不一致')

    key_to_emb = {k: seq_emb[i] for i, k in enumerate(seq_keys)}
    out = np.zeros((len(train_df), seq_emb.shape[1]), dtype=np.float32)

    # 与 W5 一致的 key 生成规则
    if 'addr1' in train_df.columns:
        row_keys = train_df['card1'].astype('int64').astype('string') + '|' + train_df['addr1'].fillna(-1).astype('int64').astype('string')
    else:
        row_keys = train_df['card1']

    n_missing = 0
    for i, k in enumerate(row_keys.values):
        emb = key_to_emb.get(k)
        if emb is None:
            n_missing += 1
            continue
        out[i] = emb

    return out, n_missing


def _metrics(y_true: np.ndarray, y_score: np.ndarray):
    auc = roc_auc_score(y_true, y_score)
    aupr = average_precision_score(y_true, y_score)
    ks = ks_2samp(y_score[y_true == 1], y_score[y_true == 0]).statistic
    return auc, aupr, ks


def _safe_logit(p: np.ndarray, eps: float = 1e-6) -> np.ndarray:
    p = np.clip(p, eps, 1 - eps)
    return np.log(p / (1 - p)).astype(np.float32)


def _train_eval_residual(name: str, feats: np.ndarray, base_prob: np.ndarray, y: np.ndarray,
                         train_idx: np.ndarray, val_idx: np.ndarray,
                         epochs: int = 3, lr: float = 5e-4, batch_size: int = 4096,
                         pos_boost: float = 2.0, hardneg_boost: float = 4.0):
    """Residual training with PR-friendly sample weights.

    - pos_boost: overall weight multiplier for positive samples
    - hardneg_boost: extra weight for high-base-prob negatives (hard negatives)
    """
    # feats: (N, D) ; base_prob: (N,)
    Xtr_np = feats[train_idx].astype(np.float32)
    ytr_np = y[train_idx].astype(np.float32)
    basep_np = base_prob[train_idx].astype(np.float32)

    Xtr = torch.tensor(Xtr_np, dtype=torch.float32)
    ytr = torch.tensor(ytr_np, dtype=torch.float32)
    base_tr = torch.tensor(_safe_logit(basep_np), dtype=torch.float32)

    Xva = torch.tensor(feats[val_idx], dtype=torch.float32)
    yva = torch.tensor(y[val_idx], dtype=torch.float32)

    # sample weights: emphasize positives + hard negatives (high baseline score but y=0)
    w = np.ones_like(ytr_np, dtype=np.float32)
    w[ytr_np == 1] *= pos_boost

    neg_mask = (ytr_np == 0)
    if neg_mask.any():
        # focus on top 1% highest-scoring negatives
        thr = float(np.quantile(basep_np[neg_mask], 0.99))
        hard = neg_mask & (basep_np >= thr)
        w[hard] *= hardneg_boost

    wtr = torch.tensor(w, dtype=torch.float32)

    ds = TensorDataset(Xtr, ytr, base_tr, wtr)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=True)

    model = ResidualFusion(feat_dim=feats.shape[1], hidden_dim=128).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    # use elementwise loss, then apply sample weights
    bce = nn.BCEWithLogitsLoss(reduction='none')

    for _ in range(epochs):
        model.train()
        for xb, yb, lb, wb in dl:
            xb, yb, lb, wb = xb.to(device), yb.to(device), lb.to(device), wb.to(device)
            delta = model(xb)
            logits = lb + delta
            loss = (bce(logits, yb) * wb).mean()
            opt.zero_grad()
            loss.backward()
            opt.step()

    model.eval()
    with torch.no_grad():
        delta_va = model(Xva.to(device)).cpu().numpy()
        logits = _safe_logit(base_prob[val_idx]) + delta_va
        scores = 1 / (1 + np.exp(-logits))

    return _metrics(yva.cpu().numpy(), scores)


# ===== 对齐三路输入到 train 每一行 =====

oof_preds = np.load(f'{CUSTOM_DIR}/oof_preds_baseline.npy').astype(np.float32)
assert oof_preds.shape[0] == len(train), f'oof_preds 长度({oof_preds.shape[0]}) != train 行数({len(train)})'

y_all = train['isFraud'].values.astype(np.int64)

gnn_train, gnn_missing = _align_gnn_to_train(train, gnn_embeddings.astype(np.float32), CUSTOM_DIR)
seq_train, seq_missing = _align_seq_to_train(train, seq_embeddings.astype(np.float32), seq_keys)

n = len(train)
print('\n=== 对齐覆盖率诊断 ===')
print(f'GNN: matched {n-gnn_missing:,}/{n:,} = {(1-gnn_missing/n)*100:.2f}%')
print(f'Seq: matched {n-seq_missing:,}/{n:,} = {(1-seq_missing/n)*100:.2f}%')

# Residual：只让 GNN/Seq 提供修正项，baseline 作为保底
F_gnn = gnn_train
F_seq = seq_train
# 当前实验中 Seq 分支增益不稳定：Full 默认仅用 GNN（更稳）；如需也可改回 concatenate
F_full = gnn_train

rng = np.random.RandomState(42)
perm = rng.permutation(len(train))
cut = int(len(train) * 0.8)
tr_idx, va_idx = perm[:cut], perm[cut:]

base_auc, base_aupr, base_ks = _metrics(y_all[va_idx], oof_preds[va_idx])

# 小网格：只调 pos/hardneg 权重，避免反复重训上游模型
settings = [
    {'pos_boost': 2.0, 'hardneg_boost': 4.0},
    {'pos_boost': 3.0, 'hardneg_boost': 4.0},
    {'pos_boost': 2.0, 'hardneg_boost': 6.0},
    {'pos_boost': 1.5, 'hardneg_boost': 4.0},
    {'pos_boost': 2.0, 'hardneg_boost': 2.0},
]

results = []
for s in settings:
    auc_gnn, aupr_gnn, ks_gnn = _train_eval_residual('+GNN', F_gnn, oof_preds, y_all, tr_idx, va_idx,
                                                     pos_boost=s['pos_boost'], hardneg_boost=s['hardneg_boost'])
    auc_seq, aupr_seq, ks_seq = _train_eval_residual('+Seq', F_seq, oof_preds, y_all, tr_idx, va_idx,
                                                     pos_boost=s['pos_boost'], hardneg_boost=s['hardneg_boost'])
    auc_full, aupr_full, ks_full = _train_eval_residual('Full', F_full, oof_preds, y_all, tr_idx, va_idx,
                                                        pos_boost=s['pos_boost'], hardneg_boost=s['hardneg_boost'])
    results.append((s['pos_boost'], s['hardneg_boost'], auc_gnn, aupr_gnn, ks_gnn, auc_seq, aupr_seq, ks_seq, auc_full, aupr_full, ks_full))

# 以 Full 的 AUC-PR 选最优
best = max(results, key=lambda r: r[9])
(best_pos, best_hn,
 auc_gnn, aupr_gnn, ks_gnn,
 auc_seq, aupr_seq, ks_seq,
 auc_full, aupr_full, ks_full) = best

print(f"Best setting by Full AUC-PR: pos_boost={best_pos}, hardneg_boost={best_hn}")
print('Fusion 训练完成。')

In [ ]:
# 消融实验对比表（展示 Full AUC-PR 最优设置下的结果）
print('\n=== 消融实验结果 (val split) ===')
print(f'Best weights: pos_boost={best_pos}, hardneg_boost={best_hn}')
print(f'{"方案":<20} {"AUC-ROC":<12} {"AUC-PR":<12} {"KS":<10}')
print('-' * 54)
print(f'{"Baseline (LightGBM)":<20} {base_auc:<12.4f} {base_aupr:<12.4f} {base_ks:<10.4f}')
print(f'{"+GNN":<20} {auc_gnn:<12.4f} {aupr_gnn:<12.4f} {ks_gnn:<10.4f}')
print(f'{"+Seq":<20} {auc_seq:<12.4f} {aupr_seq:<12.4f} {ks_seq:<10.4f}')
print(f'{"Full (三路融合)":<20} {auc_full:<12.4f} {aupr_full:<12.4f} {ks_full:<10.4f}')

print('\n(All settings)')
print(f'{"pos":<6} {"hardneg":<8} {"Full_AUPR":<10} {"Full_AUC":<10} {"Full_KS":<10}')
for r in results:
    pos, hn = r[0], r[1]
    fa, fpr, fks = r[8], r[9], r[10]
    print(f'{pos:<6} {hn:<8} {fpr:<10.4f} {fa:<10.4f} {fks:<10.4f}')

## Next: 导出产出物（用于下载/报告）

运行完上面的实验后，建议把关键信息与模型文件打包导出，方便下载与复现：

- `graphsage_best.pt`
- `transformer_best.pt`
- `gnn_embeddings.npy`
- `seq_embeddings.npy`
- 最优融合权重与消融结果（AUC-ROC / AUC-PR / KS）

In [ ]:
import json, os, zipfile, time

# 汇总本次关键结果（这些变量来自上面的 W6 输出格）
summary = {
    'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
    'best_weights': {
        'pos_boost': float(best_pos),
        'hardneg_boost': float(best_hn),
    },
    'metrics_val_split': {
        'baseline': {'auc': float(base_auc), 'aupr': float(base_aupr), 'ks': float(base_ks)},
        'gnn': {'auc': float(auc_gnn), 'aupr': float(aupr_gnn), 'ks': float(ks_gnn)},
        'seq': {'auc': float(auc_seq), 'aupr': float(aupr_seq), 'ks': float(ks_seq)},
        'full': {'auc': float(auc_full), 'aupr': float(aupr_full), 'ks': float(ks_full)},
    },
}

out_dir = '/kaggle/working/exports'
os.makedirs(out_dir, exist_ok=True)

summary_path = os.path.join(out_dir, 'experiment_summary.json')
with open(summary_path, 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

# 收集文件
files = []
for p in [
    'graphsage_best.pt',
    'transformer_best.pt',
    'gnn_embeddings.npy',
    'seq_embeddings.npy',
    summary_path,
]:
    if os.path.exists(p):
        files.append(p)
    else:
        # 这些文件可能在 /kaggle/working 下
        wp = os.path.join('/kaggle/working', p)
        if os.path.exists(wp):
            files.append(wp)

zip_path = os.path.join('/kaggle/working', 'exports.zip')
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for p in files:
        arc = os.path.join('exports', os.path.basename(p))
        z.write(p, arcname=arc)

print('Wrote:', summary_path)
print('Zipped:', zip_path)
print('Included files:', [os.path.basename(p) for p in files])